# 03 - ECG Preprocessing And Scalogram Generation

This notebook implements the first deep-feature phase:

1. read raw `.mat/.hea` ECG records,
2. parse labels and metadata,
3. bandpass-filter selected ECG leads,
4. convert leads I, II, and V5 into a 3-channel CWT scalogram image,
5. save image files and a manifest CSV for EfficientNet feature extraction.

Start with a small sample. Once the output images look correct, increase `MAX_RECORDS`.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
from tqdm.auto import tqdm

PROJECT_DIR = Path("..").resolve()
WORKSPACE_DIR = PROJECT_DIR.parents[1]
SRC_DIR = PROJECT_DIR / "src"
RAW_DATASET_DIR = WORKSPACE_DIR / "Jennie mams thing finally staritng" / "Dataset" / "WFDBRecords"

OUTPUT_DIR = PROJECT_DIR / "outputs" / "deep_features"
SCALOGRAM_DIR = OUTPUT_DIR / "scalograms_3ch_224"
MANIFEST_DIR = OUTPUT_DIR / "manifests"

for path in [SCALOGRAM_DIR, MANIFEST_DIR]:
    path.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(SRC_DIR))

from ecg_io import LABEL_MAP, discover_records, load_ecg_mat
from scalogram import DEFAULT_LEAD_INDICES, DEFAULT_LEAD_NAMES, build_3channel_scalogram, save_scalogram_png

print("Project:", PROJECT_DIR)
print("Raw dataset:", RAW_DATASET_DIR)
print("Scalogram output:", SCALOGRAM_DIR)


## Configuration

Use `MAX_RECORDS` for quick testing. Recommended:

- `100` for visual smoke test,
- `1000` for first EfficientNet experiment,
- `None` for full dataset after everything works.


In [ ]:
MAX_RECORDS = 300
TARGET_SIZE = (224, 224)  # EfficientNet-B0 prototype size. Use (380, 380) for EfficientNet-B4 final.
OVERWRITE_IMAGES = False

records = discover_records(RAW_DATASET_DIR, limit=MAX_RECORDS)
manifest_preview = pd.DataFrame([r.__dict__ for r in records])
manifest_preview["dx_codes"] = manifest_preview["dx_codes"].apply(lambda codes: ",".join(codes))

print(f"Discovered records: {len(records):,}")
display(manifest_preview[["record_id", "label", "label_name", "age", "sex", "sampling_frequency", "n_leads", "n_samples"]].head())
display(manifest_preview["label_name"].value_counts().rename_axis("label_name").reset_index(name="count"))


## Generate 3-Channel Scalograms

Each image channel represents a clinically useful lead:

- channel 0: Lead I,
- channel 1: Lead II,
- channel 2: Lead V5.


In [ ]:
rows = []
failed = []

for record in tqdm(records, desc="Generating scalograms"):
    output_path = SCALOGRAM_DIR / f"{record.record_id}.png"

    try:
        if OVERWRITE_IMAGES or not output_path.exists():
            ecg = load_ecg_mat(record.mat_path)
            image = build_3channel_scalogram(
                ecg,
                fs=record.sampling_frequency,
                lead_indices=DEFAULT_LEAD_INDICES,
                target_size=TARGET_SIZE,
            )
            save_scalogram_png(image, output_path)

        rows.append({
            "record_id": record.record_id,
            "mat_path": str(record.mat_path),
            "hea_path": str(record.hea_path),
            "image_path": str(output_path),
            "label": record.label,
            "label_name": record.label_name,
            "age": record.age,
            "sex": record.sex,
            "dx_codes": ",".join(record.dx_codes),
            "sampling_frequency": record.sampling_frequency,
            "n_leads": record.n_leads,
            "n_samples": record.n_samples,
            "lead_indices": ",".join(map(str, DEFAULT_LEAD_INDICES)),
            "lead_names": ",".join(DEFAULT_LEAD_NAMES),
            "image_height": TARGET_SIZE[1],
            "image_width": TARGET_SIZE[0],
        })
    except Exception as exc:
        failed.append({"record_id": record.record_id, "error": str(exc)})

scalogram_manifest = pd.DataFrame(rows)
failed_df = pd.DataFrame(failed)

manifest_path = MANIFEST_DIR / "scalogram_manifest_3ch_224.csv"
failed_path = MANIFEST_DIR / "scalogram_failed_records.csv"

scalogram_manifest.to_csv(manifest_path, index=False)
failed_df.to_csv(failed_path, index=False)

print(f"Saved manifest: {manifest_path}")
print(f"Generated/available images: {len(scalogram_manifest):,}")
print(f"Failed records: {len(failed_df):,}")
display(scalogram_manifest.head())


## Visual Quality Check

This cell displays one sample image from each class when available. The image should show structured time-frequency bands, not a blank or single-color square.


In [ ]:
from PIL import Image

examples = (
    scalogram_manifest
    .sort_values("record_id")
    .groupby("label_name", as_index=False)
    .head(1)
)

fig, axes = plt.subplots(1, len(examples), figsize=(5 * len(examples), 4))
if len(examples) == 1:
    axes = [axes]

for ax, (_, row) in zip(axes, examples.iterrows()):
    img = Image.open(row["image_path"])
    ax.imshow(img)
    ax.set_title(f"{row['record_id']} | {row['label_name']}")
    ax.axis("off")

plt.tight_layout()
plt.show()


## Next Output

The next notebook, `04_efficientnet_embeddings.ipynb`, reads:

`outputs/deep_features/manifests/scalogram_manifest_3ch_224.csv`

and extracts EfficientNet embeddings from these saved images.
